# API参考: `HydrologicalModel`

**类路径:** `hydro_model.model.HydrologicalModel`

## 目的
`HydrologicalModel` 是一个容器类，它将一个**产流模块（Runoff Module）**和一个**汇流模块（Routing Module）**组合在一起，形成一个完整的、概念性的集总式水文模型。它本身不包含复杂的水文物理计算，而是负责调用其包含的子模块，并按正确的顺序传递数据。

## 继承关系
`HydrologicalModel` 继承自 `common.base_model.BaseModelComponent`，因此它可以作为一个标准组件被添加到`SimulationController`中。

## `__init__(self, name, runoff_module, routing_module)`

构造函数用于创建一个`HydrologicalModel`的实例。

**参数:**
- `name` (str): (继承自`BaseModelComponent`) 组件的唯一名称。
- `runoff_module` (BaseRunoffModule): 一个产流模块的实例（例如 `SCSCurveNumberModule` 或 `SimpleRunoffModule`）。
- `routing_module` (BaseRoutingModule): 一个汇流模块的实例（例如 `SimpleRouting` 或 `MuskingumRouting`）。

### 示例

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))

from hydro_model.model import HydrologicalModel
from hydro_model.runoff import SCSCurveNumberModule
from hydro_model.routing import SimpleRouting

# 1. 创建子模块实例
 runoff_mod = SCSCurveNumberModule(CN=75)
 routing_mod = SimpleRouting(k_q=0.8, k_s=0.2)

# 2. 创建 HydrologicalModel 实例
hydro_model = HydrologicalModel(
    name="my_catchment",
    runoff_module=runoff_mod,
    routing_module=routing_mod
)

print(f"成功创建 HydrologicalModel: '{hydro_model.name}'")

## `run(self, rainfall, pet)`

这是模型的核心方法，它执行一个时间步的计算。它不继承自`BaseModelComponent`的`step`方法，因为它有自己独特的输入（降雨和蒸发）。

**逻辑流程:**
1.  调用其`runoff_module`的`run`方法，传入`rainfall`和`pet`，计算出该时间步的产流量（有效径流深）。
2.  将上一步计算出的产流量，作为输入传递给其`routing_module`的`run`方法，计算出经过汇流演算后的最终出流量。
3.  更新自身的`outflow`属性。
4.  返回最终的出流量。

**参数:**
- `rainfall` (float): 当前时间步的总降雨量 (mm)。
- `pet` (float): 当前时间步的潜在蒸散发量 (mm)。

**返回:**
- (float): 当前时间步的最终出流量 (mm)。

### 示例

In [ ]:
rainfall_at_t = 20.0
pet_at_t = 1.0

outflow_at_t = hydro_model.run(rainfall=rainfall_at_t, pet=pet_at_t)

print(f"给定降雨 {rainfall_at_t}mm, PET {pet_at_t}mm:")
print(f"  - 计算得到的产流量: {hydro_model.runoff_module.last_runoff:.3f} mm") # 假设 runoff_module 记录了上次结果
print(f"  - 最终的出流量: {outflow_at_t:.3f} mm")

## `step(self, inflows: dict, dt: float)`

这是`HydrologicalModel`作为`BaseModelComponent`所实现的接口方法。它使得`HydrologicalModel`可以被`SimulationController`调用。

**逻辑流程:**
1.  它会从`inflows`字典中查找`'rainfall'`和`'pet'`这两个键，作为全局输入。
2.  然后，它调用自身的`run(rainfall, pet)`方法来执行计算。

### 示例

In [ ]:
# 当被控制器调用时，inflows字典会由控制器提供
inflows_from_controller = {
    'rainfall': 20.0,
    'pet': 1.0
}

hydro_model.step(inflows=inflows_from_controller, dt=3600) # dt 在这里不被使用

print(f"通过 step() 方法得到的 outflow: {hydro_model.get_outflow():.3f} mm")